In [1]:
import os, glob, json
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# DATA_PATH = '/kaggle/input/tashkeel-dataset/dataset/'
DATA_PATH = 'dataset/'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')
OUTPUT_MODEL_PATH = '/kaggle/working/bilstm_diac_pytorch_with_der.pt'
# MAXLEN = 500
# EMBED_DIM = 25
# LSTM_UNITS = 256
# FF_UNITS = 512
# DROPOUT = 0.5
# BATCH_SIZE = 64
EPOCHS = 20
MAXLEN = 500
EMBED_DIM = 64
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 256
EPOCHS = 40


PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'


Using device: cpu


In [2]:

# =========================
# Robust Arabic preprocessing for diacritization
# =========================

import unicodedata

# -------------------------
# Unicode helpers
# -------------------------
def is_combining(ch):
    """Arabic diacritic mark"""
    return unicodedata.category(ch) == "Mn"


def is_arabic_letter(ch):
    """True Arabic letters only (no punctuation, no digits)"""
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return False
    return "ARABIC" in name and unicodedata.category(ch).startswith("L")


def strip_diacritics(text):
    """Remove ALL existing diacritics"""
    return "".join(ch for ch in text if not is_combining(ch))


# -------------------------
# Normalize Arabic (recommended)
# -------------------------
# def normalize_arabic(text):
#     return (
#         text.replace("أ","ا")
#             .replace("إ","ا")
#             .replace("آ","ا")
#             .replace("ى","ي")
#             .replace("ؤ","و")
#             .replace("ئ","ي")
#     )
def normalize_arabic(text):
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    return text
# -------------------------
# Split base + diacritics
# -------------------------
def split_char_diacritic_pairs(sentence):
    """
    Input:  string with Arabic + diacritics
    Output: [(base_char, diacritics)]
    """
    pairs = []
    base = None
    diacs = ""

    for ch in sentence:
        if is_combining(ch):
            if base is None:
                base = "<UNK_BASE>"
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ""

    if base is not None:
        pairs.append((base, diacs))

    return pairs


# -------------------------
# Token + label generation
# -------------------------
def placeholder_transform_pairs(
    pairs,
    placeholder=PLACEHOLDER,
):
    """
    Converts (base, diacritics) → tokens + labels
    """
    tokens = []
    labels = []

    for base, d in pairs:

        # SPACE
        if base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append("")
            continue

        # Arabic letter (keep)
        if is_arabic_letter(base) or base == "ـ":
            tokens.append(base)
            labels.append(d)
            continue

        # EVERYTHING ELSE → NONAR placeholder
        tokens.append(placeholder)
        labels.append("")

    return tokens, labels


# -------------------------
# FULL preprocessing pipeline
# -------------------------
def preprocess_sentence(sentence):
    """
    SAFE preprocessing for training & inference
    """
    sentence = strip_diacritics(sentence)
    sentence = normalize_arabic(sentence)

    pairs = split_char_diacritic_pairs(sentence)
    tokens, labels = placeholder_transform_pairs(pairs)

    return tokens, labels


In [3]:


class BiLSTM_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # UPDATED: We use a single LSTM with num_layers=2
        # 'dropout=dropout' here applies dropout between Layer 1 and Layer 2
        self.bilstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=lstm_units,
            num_layers=2,                # <--- Changed from 1 to 2
            batch_first=True,
            bidirectional=True,
            dropout=dropout              # Applies dropout between the 2 LSTM layers
        )

        # This dropout applies to the output of the LAST layer before the feed-forward network
        self.dropout_final = nn.Dropout(dropout)

        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()

    def forward(self, x, lengths=None):
        emb = self.embedding(x)

        if lengths is not None:
            # Pack padded sequence
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            
            # Pass through the 2-layer BiLSTM
            packed_out, _ = self.bilstm(packed)
            
            # Unpack
            out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            # Validation/Inference (no packing needed if batch size is 1 or fixed length)
            out, _ = self.bilstm(emb)

        # Apply dropout to the final LSTM output
        out = self.dropout_final(out)

        # Feedforward layers
        ff = self.relu(self.ff1(out))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        
        return logits

In [ ]:
# =========================
# Inference → CSV + RAW/DIAC printing (CHUNKED, SAFE, PREPROCESSING-ALIGNED)
# =========================

import os
import csv
import pickle
import unicodedata
import torch

# -------------------------
# Load external label mapping
# -------------------------
PICKLE_LABEL_MAP_PATH = "diacritic2id.pickle"

with open(PICKLE_LABEL_MAP_PATH, "rb") as f:
    external_diac2label = pickle.load(f)

print("External diacritic → label mapping:")
for k, v in external_diac2label.items():
    print(f"'{k}': {v}")

# -------------------------
# Load trained model
# -------------------------
MODEL_PATH = "Output/1_bilstm.pt"
ckpt = torch.load(MODEL_PATH, map_location=device)

MODEL2_PATH = "Output/bilstm_Train_with_Val.pt"
ckpt2 = torch.load(MODEL2_PATH, map_location=device)

char2idx = ckpt["char2idx"]
diac2idx = ckpt["diac2idx"]
idx2diac = {v: k for k, v in diac2idx.items()}

pad_idx = char2idx[PAD_TOKEN]

model = BiLSTM_Diac(
    vocab_size=len(char2idx),
    emb_dim=EMBED_DIM,
    lstm_units=LSTM_UNITS,
    ff_units=FF_UNITS,
    num_labels=len(diac2idx),
    pad_idx=pad_idx,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()


# -------------------------
# Inference preprocessing (ALIGNED with training)
# -------------------------
def prepare_inference_raw_line(raw_line):
    """
    Preprocess inference line using SAME logic as training,
    while tracking original non-Arabic characters for reconstruction.
    """
    raw_line = strip_diacritics(raw_line)
    raw_line = normalize_arabic(raw_line)

    pairs = split_char_diacritic_pairs(raw_line)
    tokens, _ = placeholder_transform_pairs(pairs)

    original_nonar = []
    pos = 0
    for ch in raw_line:
        if not (is_arabic_letter(ch) or ch == "ـ" or ch.isspace()):
            original_nonar.append((pos, ch))
        if ch.isspace() or is_arabic_letter(ch) or ch == "ـ":
            pos += 1

    return tokens, original_nonar


def training_label_to_external_label(train_label_id):
    diac = idx2diac.get(int(train_label_id), "")
    if diac in ("<NONE>", "<PAD_LABEL>", "<OTHER>", None):
        diac = ""
    return external_diac2label.get(diac, external_diac2label.get("", 0))


def reconstruct_diacritized_line(tokens, pred_ids, original_nonar):
    out = []
    nonar_map = {pos: ch for pos, ch in original_nonar}

    for i, (tok, pid) in enumerate(zip(tokens, pred_ids)):
        if i in nonar_map:
            out.append(nonar_map[i])
            continue

        if tok == SPACE_TOKEN:
            out.append(" ")
            continue

        diac = idx2diac.get(int(pid), "")
        if diac in ("<NONE>", "<PAD_LABEL>", "<OTHER>"):
            diac = ""

        out.append(unicodedata.normalize("NFC", tok + diac))

    return "".join(out)


def is_gold_arabic_char(ch):
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    if ch == "ـ":
        return False
    if not unicodedata.category(ch).startswith("L"):
        return False
    try:
        return "ARABIC" in unicodedata.name(ch)
    except ValueError:
        return False

# -------------------------
# Chunked inference (BiLSTM-safe)
# -------------------------
WINDOW = MAXLEN
OVERLAP = 50
STRIDE = WINDOW - OVERLAP - 2


def infer_tokens_chunked(tokens):
    all_pred_ids = []
    pos = 0

    while pos < len(tokens):
        chunk = tokens[pos : pos + STRIDE]

        seq_mod = [SOS_TOKEN] + chunk + [EOS_TOKEN]
        x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
        x_ids += [char2idx[PAD_TOKEN]] * (WINDOW - len(x_ids))

        x_tensor = torch.tensor([x_ids], dtype=torch.long).to(device)
        lengths = torch.tensor([len(seq_mod)], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(x_tensor, lengths)
            pred = torch.argmax(logits, dim=-1)[0]

        pred = pred[1 : 1 + len(chunk)]
        all_pred_ids.extend(pred.cpu().tolist())

        pos += STRIDE

    return all_pred_ids[:len(tokens)]

# -------------------------
# Inference → CSV + printing
# -------------------------
def infer_text_to_csv(raw_text, output_csv_path):
    rows = []
    global_id = 0

    for raw_line in raw_text.splitlines():
        if not raw_line.strip():
            print()
            continue

        tokens, original_nonar = prepare_inference_raw_line(raw_line)
        pred_ids = infer_tokens_chunked(tokens)

        diac_line = reconstruct_diacritized_line(tokens, pred_ids, original_nonar)

        print("RAW:  ", raw_line)
        print("DIAC: ", diac_line)
        print()

        for tok, pid in zip(tokens, pred_ids):
            if not is_gold_arabic_char(tok):
                continue

            ext_label = training_label_to_external_label(pid)
            rows.append((global_id, ext_label))
            global_id += 1

    with open(output_csv_path, "w", newline="", encoding="utf8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "label"])
        writer.writerows(rows)

    print(f"Saved CSV predictions to: {output_csv_path}")

# -------------------------
# Run
# -------------------------
test_file_path = "dataset_no_diacritics.txt"

with open(test_file_path, "r", encoding="utf8") as f:
    raw_text = f.read()

infer_text_to_csv(raw_text, "predictions.csv")


External diacritic → label mapping:
'َ': 0
'ً': 1
'ُ': 2
'ٌ': 3
'ِ': 4
'ٍ': 5
'ْ': 6
'ّ': 7
'َّ': 8
'ًّ': 9
'ُّ': 10
'ٌّ': 11
'ِّ': 12
'ٍّ': 13
'': 14


C:\Users\menna\AppData\Local\Temp\ipykernel_109292\1947470262.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)
C:\User

RAW:   ما أحسن ما شكا أمره بين أضعاف مدحه
DIAC:  مَا احْسَنَّ مَا شَكَا امْرَهُ بَيْنَ اضْعَافِ مَدْحِهِ

RAW:   فقد نجا كونوا علماء وجالسوا العلماء
DIAC:  فَقَدْ نَجَا كَوْنُوا عُلَمَاءَ وَجَالَسُوا الْعُلَمَاءَ

RAW:   وقال بزرجمهر من العلم أن لا تحتقر شيئا من العلم
DIAC:  وَقَالَ بِزِرْجَمْهِرٍ مِنْ الْعِلْمِ انْ لَا تَحْتَقِرُ شَيْيًّا مِنْ الْعِلْمِ

RAW:   فينبغي لمن استدل بفطرته على استحسان الفضائل
DIAC:  فَيَنْبَغِي لِمَنْ اسْتَدَلَّ بِفِطْرَتِهِ عَلَيَّ اسْتِحْسَانُ الْفَضَايِلِ

RAW:   كدر معروفا امتنان
DIAC:  كَدْرٍ مَعْرُوفًا امْتِنَانِ

RAW:   وفي الأذكار عن بعضهم : يسن لمن لم يتمكن منها لحدث أو شغل أو نحوه أن يقول ذلك أربعا ، قال المصنف : إنه لا بأس به .
DIAC:  وَفِي الِاذْكَارِ عَنْ بَعْضِهِمْ : يُسَنُّ لِمَنْ لَمْ يَتَمَكَّنْ مِنْهَا لِحَدَثٍ اوْ شَغِلَ اوْ نَحْوَهُ انْ يَقُولُ ذَلِكَ ارْبَعَا،<NONAR> قَالَ الْمُصَنِّ: <NONAR> انَّهُ لَا بَاسَ .هِ <NONAR>

RAW:   ناشدت منظمة الأغذية والزراعة الجهات المانحة والمتبرعين لجمع أكثر من 40 مليون دولار على وجه السرعة لتأمين الب